# FINARY - Insight Profile Model (Deep Learning)

Notebook ini membangun model multi-output untuk:
1. Prediksi saldo bulan depan (`balance_output`, regresi).
2. Prediksi warning finansial (`warning_output`, klasifikasi).

Ketentuan yang dipenuhi:
- TensorFlow Functional API.
- Komponen kustom lanjutan: `MetricsGuardCallback` (Custom Callback).
- Export model `.keras` dan `SavedModel`.
- Inference sederhana.
- Siap dipakai REST API FastAPI.

In [1]:
import os
import json
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("FINARY_FIXED_FINAL_DATASET.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [2]:
df = pd.read_csv(DATA_PATH)

# Konversi boolean ke integer agar konsisten untuk scaler/model.
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

# Target warning berbasis risk rules bisnis.
warning_label = (
    (df["net_cash_flow"] < 0)
    | (df["debt_ratio_flag"] == 1)
    | (df["low_emergency_flag"] == 1)
).astype("float32")

# Proxy target saldo bulan depan (karena label saldo aktual bulan depan tidak tersedia di dataset).
next_month_balance_proxy = df["emergency_fund"] + df["net_cash_flow"]

balance_min = float(next_month_balance_proxy.min())
balance_max = float(next_month_balance_proxy.max())

balance_target = ((next_month_balance_proxy - balance_min) / (balance_max - balance_min)).astype("float32")

X = df.astype("float32")
feature_columns = X.columns.tolist()

print("Data shape:", X.shape)
print("Warning rate:", float(warning_label.mean()))
print("Balance target range:", float(balance_target.min()), float(balance_target.max()))

Data shape: (3000, 43)
Warning rate: 0.5989999771118164
Balance target range: 0.0 1.0


In [3]:
X_train, X_temp, yb_train, yb_temp, yw_train, yw_temp = train_test_split(
    X.values,
    balance_target.values,
    warning_label.values,
    test_size=0.30,
    random_state=SEED,
    stratify=warning_label.values,
)

X_val, X_test, yb_val, yb_test, yw_val, yw_test = train_test_split(
    X_temp,
    yb_temp,
    yw_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=yw_temp,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")
with open(ARTIFACT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)
with open(ARTIFACT_DIR / "target_stats.json", "w", encoding="utf-8") as f:
    json.dump({"balance_min": balance_min, "balance_max": balance_max}, f, indent=2)

print("Train/Val/Test:", X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

Train/Val/Test: (2100, 43) (450, 43) (450, 43)


In [9]:
class MetricsGuardCallback(tf.keras.callbacks.Callback):
    """Stop training once project threshold metrics are reached."""

    def __init__(self, min_accuracy=0.85, max_mae=0.02):
        super().__init__()
        self.min_accuracy = min_accuracy
        self.max_mae = max_mae

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get("val_warning_output_accuracy", 0.0)
        val_mae = logs.get("val_balance_output_mae", 1.0)
        if val_acc >= self.min_accuracy and val_mae <= self.max_mae:
            print(f"Threshold reached at epoch {epoch + 1}: val_acc={val_acc:.4f}, val_mae={val_mae:.4f}")
            self.model.stop_training = True


inputs = tf.keras.Input(shape=(X_train_scaled.shape[1],), name="finance_features")
shared = tf.keras.layers.Dense(64, activation="relu")(inputs)
shared = tf.keras.layers.Dense(32, activation="relu")(shared)

balance_output = tf.keras.layers.Dense(1, activation="linear", name="balance_output")(shared)
warning_hidden = tf.keras.layers.Dense(32, activation="relu")(shared)
warning_output = tf.keras.layers.Dense(1, activation="sigmoid", name="warning_output")(warning_hidden)

model = tf.keras.Model(inputs=inputs, outputs=[balance_output, warning_output], name="finary_multitask_model")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "balance_output": "mse",
        "warning_output": tf.keras.losses.BinaryCrossentropy(),
    },
    loss_weights={"balance_output": 0.7, "warning_output": 0.3},
    metrics={
        "balance_output": [tf.keras.metrics.MeanAbsoluteError(name="mae")],
        "warning_output": [tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    },
)

model.summary()

Model: "finary_multitask_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ finance_features    │ (None, 43)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      2,816 │ finance_features… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      2,080 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      1,056 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ balance_output      │ (None, 1)         │         33 │ dense_4[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ warning_output      │ (None, 1)         │         33 │ dense_5[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,018 (23.51 KB)

 Trainable params: 6,018 (23.51 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
callbacks = [
    MetricsGuardCallback(min_accuracy=0.85, max_mae=0.02),
]

history = model.fit(
    X_train_scaled,
    {
        "balance_output": yb_train,
        "warning_output": yw_train,
    },
    validation_data=(
        X_val_scaled,
        {
            "balance_output": yb_val,
            "warning_output": yw_val,
        },
    ),
    epochs=300,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - balance_output_loss: 0.1100 - balance_output_mae: 0.2457 - loss: 0.2791 - warning_output_accuracy: 0.5948 - warning_output_loss: 0.6726 - val_balance_output_loss: 0.0580 - val_balance_output_mae: 0.1773 - val_loss: 0.2225 - val_warning_output_accuracy: 0.6556 - val_warning_output_loss: 0.6053
Epoch 2/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - balance_output_loss: 0.0383 - balance_output_mae: 0.1523 - loss: 0.1974 - warning_output_accuracy: 0.7257 - warning_output_loss: 0.5679 - val_balance_output_loss: 0.0451 - val_balance_output_mae: 0.1501 - val_loss: 0.1778 - val_warning_output_accuracy: 0.7711 - val_warning_output_loss: 0.4765
Epoch 3/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - balance_output_loss: 0.0262 - balance_output_mae: 0.1275 - loss: 0.1429 - warning_output_accuracy: 0.8395 - warning_output_loss: 0.4145 - val_balance_output_loss: 0.0347 - val_balance_output_mae: 0.1353 - val_loss: 0.1214 - val_warning_output_accuracy: 

In [13]:
evaluation = model.evaluate(
    X_test_scaled,
    {
        "balance_output": yb_test,
        "warning_output": yw_test,
    },
    verbose=0,
    return_dict=True,
)

metrics_summary = {
    "test_balance_mae": float(evaluation["balance_output_mae"]),
    "test_warning_accuracy": float(evaluation["warning_output_accuracy"]),
}
print(json.dumps(metrics_summary, indent=2))

if metrics_summary["test_warning_accuracy"] < 0.85:
    raise ValueError("Akurasi warning < 0.85. Model belum memenuhi ketentuan.")
if metrics_summary["test_balance_mae"] > 0.02:
    raise ValueError("MAE balance > 0.02. Model belum memenuhi ketentuan.")

print("Model memenuhi ketentuan minimum proyek.")

{
  "test_balance_mae": 0.019186394289135933,
  "test_warning_accuracy": 0.9777777791023254
}
Model memenuhi ketentuan minimum proyek.


In [15]:
keras_model_path = ARTIFACT_DIR / "finary_multitask_model.keras"
savedmodel_path = ARTIFACT_DIR / "finary_multitask_savedmodel"

model.save(keras_model_path)
model.export(savedmodel_path)

print("Saved .keras model to:", keras_model_path)
print("Saved SavedModel to:", savedmodel_path)

INFO:tensorflow:Assets written to: artifacts\finary_multitask_savedmodel\assets


INFO:tensorflow:Assets written to: artifacts\finary_multitask_savedmodel\assets


Saved artifact at 'artifacts\finary_multitask_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 43), dtype=tf.float32, name='finance_features')
Output Type:
  List[TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)]
Captures:
  3029163490768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029129885648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029163490960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181104784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181106128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181105936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181107472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181105360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181105168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3029181104400: TensorS

In [2]:
# %% [8. TEST INFERENCE INSIGHT]
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

# 1. SETUP & LOAD
ARTIFACT_DIR = Path("artifacts")
loaded_model = tf.keras.models.load_model(ARTIFACT_DIR / "finary_multitask_model.keras")
loaded_scaler = joblib.load(ARTIFACT_DIR / "scaler.joblib")

with open(ARTIFACT_DIR / "feature_columns.json", "r") as f:
    feature_columns = json.load(f)
with open(ARTIFACT_DIR / "target_stats.json", "r") as f:
    target_stats = json.load(f)
    
balance_min = float(target_stats["balance_min"])
balance_max = float(target_stats["balance_max"])
UNIT_SCALE = 2500.0  

# 2. LOGIKA REKOMENDASI (Disesuaikan dengan API)
def build_recommendations_api_style(features_dict, warning_prob):
    recs = []
    if features_dict.get("debt_ratio_flag", 0) == 1.0: 
        recs.append("Prioritaskan pelunasan utang berbunga tinggi.")
    if features_dict.get("low_emergency_flag", 0) == 1.0: 
        recs.append("Tingkatkan emergency fund.")
    if warning_prob > 0.7: 
        recs.append("Warning tinggi: batasi transaksi non-esensial selama 2 minggu.")
    if not recs: 
        recs.append("Profil keuangan sehat.")
    return recs

# 3. FUNGSI PREDIKSI
def predict_insight_notebook_v2(payload):
    # Preprocessing (Sesuai API)
    inc = float(payload.get("income", 0)) / UNIT_SCALE
    exp = float(payload.get("expense", 0)) / UNIT_SCALE
    sav = float(payload.get("savings", 0)) / UNIT_SCALE
    tgt_sav = float(payload.get("target_tabungan", 0)) / UNIT_SCALE
    loan = float(payload.get("loan_payment", 0)) / UNIT_SCALE
    emg = float(payload.get("emergency_fund", 0)) / UNIT_SCALE
    inc_type = payload.get("income_type", "Salary")
    category = payload.get("main_category", "Utilities")

    net_cf = inc - exp
    dti = loan / inc if inc > 0 else 0.0
    buffer = emg / exp if exp > 0 else 0.0
    
    # Inisialisasi Fitur
    features = {col: 0.0 for col in feature_columns}
    features.update({
        "monthly_income": inc, 
        "monthly_expense_total": exp, 
        "actual_savings": sav,
        "budget_goal": tgt_sav, 
        "loan_payment": loan, 
        "emergency_fund": emg,
        "net_cash_flow": net_cf, 
        "savings_rate": sav / inc if inc > 0 else 0.0,
        "expense_ratio": exp / inc if inc > 0 else 0.0, 
        "debt_to_income_ratio": dti,
        "financial_buffer": buffer, 
        "savings_goal_met": 1.0 if sav >= tgt_sav else 0.0,
        "debt_ratio_flag": 1.0 if dti >= 0.35 else 0.0, 
        "low_emergency_flag": 1.0 if buffer < 1.0 else 0.0
    })
    
    # One-Hot Encoding Manual
    if f"income_type_{inc_type}" in features: features[f"income_type_{inc_type}"] = 1.0
    if f"category_{category}" in features: features[f"category_{category}"] = 1.0
    features["cash_flow_status_Positive"] = 1.0 if net_cf > 0 else 0.0
    features["cash_flow_status_Neutral"] = 1.0 if net_cf <= 0 else 0.0

    # Scaling & Predict
    df_input = pd.DataFrame([features])[feature_columns]
    X_scaled = loaded_scaler.transform(df_input.values)
    
    pred_balance_norm, pred_warning_prob = loaded_model.predict(X_scaled, verbose=0)
    
    # Post-processing
    p_bal = float(np.clip(pred_balance_norm[0][0], 0.0, None))
    p_warn = float(np.clip(pred_warning_prob[0][0], 0.0, 1.0))
    
    # Balikin ke IDR
    predicted_balance_idr = (p_bal * (balance_max - balance_min) + balance_min) * UNIT_SCALE
        
    return {
        "predicted_next_month_balance": round(predicted_balance_idr, 2),
        "warning_probability": round(p_warn, 4),
        "warning_flag": int(p_warn >= 0.5),
        "recommendations": build_recommendations_api_style(features, p_warn)
    }

# --- RUN TEST ---
sample_insight = {
    "income": 10000000,
    "expense": 6000000,
    "savings": 1000000,
    "target_tabungan": 5000000,
    "loan_payment": 0,
    "emergency_fund": 1000000,
    "income_type": "Salary",
    "main_category": "Utilities"
}

hasil = predict_insight_notebook_v2(sample_insight)
print(json.dumps(hasil, indent=2))

{
  "predicted_next_month_balance": 5147351.53,
  "warning_probability": 1.0,
  "warning_flag": 1,
  "recommendations": [
    "Tingkatkan emergency fund.",
    "Warning tinggi: batasi transaksi non-esensial selama 2 minggu."
  ]
}


## Menjalankan REST API FastAPI

Setelah notebook ini dijalankan sampai selesai (model + artifacts tersimpan), jalankan API:

```bash
uvicorn api_service:app --reload --host 0.0.0.0 --port 8000
```

Endpoint:
- `GET /health`
- `POST /predict` dengan payload `{ "features": { ...43 fitur... } }`